In [1]:
import pandas as pd
import pyarrow.parquet as pq
from datasets import load_dataset
import tempfile
import subprocess
import json
from pathlib import Path

In [2]:
ai_dev_v5_path = r"E:\!My Files\CS\Grammar\AIDev-7.6M"
swe_chat_path = r"E:\!My Files\CS\Grammar\SWE-Chat"

Look at AIDev

In [ ]:
# all_repo_df = pd.read_parquet(ai_dev_v5_path + "/all_repository.parquet")
# all_user_df = pd.read_parquet(ai_dev_v5_path + "/all_user.parquet")
# all_pull_request_df = pd.read_parquet(ai_dev_v5_path + "/all_pull_request.parquet", engine="pyarrow").head(10000)


# AIDev-pop
pull_request_df = pd.read_parquet(ai_dev_v5_path + "/pull_request.parquet")
repository_df = pd.read_parquet(ai_dev_v5_path + "/repository.parquet")
pr_commits_df = pd.read_parquet(ai_dev_v5_path + "/pr_commits.parquet")
pr_commit_details_df = pd.read_parquet(ai_dev_v5_path + "/pr_commit_details.parquet")
pr_timeline_df = pd.read_parquet(ai_dev_v5_path + "/pr_timeline.parquet")

In [4]:
pr_timeline_df

,pr_id,event,commit_id,created_at,actor,assignee,label,message
0,4019919434,committed,279466730b8b9656e4be53ad3e5e57412f68b03a,None,AlanJǜn,None,None,Add Redis queues and real CLOB order path
1,4019919434,labeled,None,2026-03-04T04:11:08Z,jvnhaoWen,None,codex,None
2,4019919434,committed,32f1334d0eecb46171ace07e10d6a54728617d80,None,AlanJǜn,None,None,Rebuild project as poly-monitor Twitter auto-t...
3,4019919434,committed,35baa16492146f2d2c5b2242b1629694c32fb9a8,None,AlanJǜn,None,None,Refocus poly-monitor to prompt-driven task con...
4,4019919434,committed,f4a633e6149a0eb12cc07737e9f0e2b4f15ecc17,None,AlanJǜn,None,None,Move decision flow into decision.py and decoup...
...,...,...,...,...,...,...,...,...
3018353,3626123470,unsubscribed,None,2025-11-15T11:16:32Z,ChrisDenton,None,None,None
3018354,3626123470,reviewed,5945f08bd5316684a622c1e2d94876d9197d97fc,None,None,None,None,None
3018355,3626123470,copilot_work_started,None,2025-11-24T11:32:58Z,rzikm,None,None,None
3018356,3626123470,review_requested,None,2025-11-24T11:54:05Z,Copilot,None,None,None


In [5]:
print(f"{pull_request_df.columns=}", "\n")
print(f"{repository_df.columns=}", "\n")
print(f"{pr_commits_df.columns=}", "\n")
print(f"{pr_commit_details_df.columns=}", "\n")
print("---\n---")
print(f"{pr_timeline_df.columns=}", "\n")

pull_request_df.columns=Index(['id', 'number', 'title', 'body', 'agent', 'user_id', 'user', 'state',
       'created_at', 'closed_at', 'merged_at', 'repo_id', 'repo_url',
       'html_url'],
      dtype='object') 

repository_df.columns=Index(['id', 'url', 'license', 'full_name', 'is_forked', 'language', 'forks',
       'stars'],
      dtype='object') 

pr_commits_df.columns=Index(['sha', 'pr_id', 'author', 'committer', 'message'], dtype='object') 

pr_commit_details_df.columns=Index(['sha', 'pr_id', 'author', 'committer', 'message', 'commit_stats_total',
       'commit_stats_additions', 'commit_stats_deletions', 'filename',
       'status', 'additions', 'deletions', 'changes', 'patch'],
      dtype='object') 

---
---
pr_timeline_df.columns=Index(['pr_id', 'event', 'commit_id', 'created_at', 'actor', 'assignee',
       'label', 'message'],
      dtype='object') 



In [6]:
# Join all pop tables - Attach PR and repository metadata to each commit-file row.
aidev_joined_df = (
    pr_commit_details_df
    .merge(
        pull_request_df,
        left_on="pr_id",
        right_on="id",
        how="left",
        suffixes=("", "_pr"),
        validate="many_to_one",
    )
    .merge(
        repository_df,
        left_on="repo_id",
        right_on="id",
        how="left",
        suffixes=("", "_repo"),
        validate="many_to_one",
    )
)
"""AIDev-v5-pop all tables joined needed to create the pre/post patch pairs to analyze"""

display(aidev_joined_df.head())
display(aidev_joined_df.columns)

,sha,pr_id,author,committer,message,commit_stats_total,commit_stats_additions,commit_stats_deletions,filename,status,...,repo_url,html_url,id_repo,url,license,full_name,is_forked,language,forks,stars
0,ea9ced880ce569a3221cb92be82467b36abcc9a4,3144088312,MontrealAI,MontrealAI,docs: emphasize setup script,16,10,6,AGENTS.md,modified,...,https://api.github.com/repos/MontrealAI/AGI-Al...,https://github.com/MontrealAI/AGI-Alpha-Agent-...,922805069,https://api.github.com/repos/MontrealAI/AGI-Al...,Apache-2.0,MontrealAI/AGI-Alpha-Agent-v0,False,Python,62,283
1,ea9ced880ce569a3221cb92be82467b36abcc9a4,3144088312,MontrealAI,MontrealAI,docs: emphasize setup script,16,10,6,alpha_factory_v1/scripts/preflight.py,modified,...,https://api.github.com/repos/MontrealAI/AGI-Al...,https://github.com/MontrealAI/AGI-Alpha-Agent-...,922805069,https://api.github.com/repos/MontrealAI/AGI-Al...,Apache-2.0,MontrealAI/AGI-Alpha-Agent-v0,False,Python,62,283
2,ea9ced880ce569a3221cb92be82467b36abcc9a4,3144088312,MontrealAI,MontrealAI,docs: emphasize setup script,16,10,6,codex/setup.sh,modified,...,https://api.github.com/repos/MontrealAI/AGI-Al...,https://github.com/MontrealAI/AGI-Alpha-Agent-...,922805069,https://api.github.com/repos/MontrealAI/AGI-Al...,Apache-2.0,MontrealAI/AGI-Alpha-Agent-v0,False,Python,62,283
3,6106a2e6bdde73d784fb39e56ebecfb999ddc3de,3726213538,raphaelscholle,raphaelscholle,Add Linux 6.x NXP V4L2 encoder pipeline with i...,437,408,29,OpenHD/ohd_common/CMakeLists.txt,modified,...,https://api.github.com/repos/OpenHD/OpenHD,https://github.com/OpenHD/OpenHD/pull/1271,161053237,https://api.github.com/repos/OpenHD/OpenHD,GPL-3.0,OpenHD/OpenHD,False,C++,387,2331
4,6106a2e6bdde73d784fb39e56ebecfb999ddc3de,3726213538,raphaelscholle,raphaelscholle,Add Linux 6.x NXP V4L2 encoder pipeline with i...,437,408,29,OpenHD/ohd_common/inc/kernel_version.h,added,...,https://api.github.com/repos/OpenHD/OpenHD,https://github.com/OpenHD/OpenHD/pull/1271,161053237,https://api.github.com/repos/OpenHD/OpenHD,GPL-3.0,OpenHD/OpenHD,False,C++,387,2331


Index(['sha', 'pr_id', 'author', 'committer', 'message', 'commit_stats_total',
       'commit_stats_additions', 'commit_stats_deletions', 'filename',
       'status', 'additions', 'deletions', 'changes', 'patch', 'id', 'number',
       'title', 'body', 'agent', 'user_id', 'user', 'state', 'created_at',
       'closed_at', 'merged_at', 'repo_id', 'repo_url', 'html_url', 'id_repo',
       'url', 'license', 'full_name', 'is_forked', 'language', 'forks',
       'stars'],
      dtype='object')

Now that we've joined the tables that we needed to get all agentic task information, including ```repo```, ```pr```, ```commit```, ```patch```, etc. information per pr, we can start to filter for our target task (CI/CD, SWE), and create target pairs to compare pre, and post- agent task. The diffs are under the ```patch``` column, but the dataset creators mentioned that some are too big and hence were not included in the dataset. Those will need to be manually downloaded from github.

We have 395,260 commit-file rows across 25,563 prs

In [3]:
AIDev_v5_pop_path = Path("Data/AIDev-v5-pop_ci-cd_filtered.parquet")

if not AIDev_v5_pop_path.is_file():
    workflow_changes_df = aidev_joined_df.loc[
        aidev_joined_df["filename"].str.startswith(".github/workflows/", na=False)
        & aidev_joined_df["filename"].str.lower().str.endswith((".yml", ".yaml"), na=False)
    ].copy()

    # Save workflow filtered AIDev-v5-pop
    workflow_changes_df.to_parquet("Data/AIDev-v5-pop_ci-cd_filtered.parquet", engine="pyarrow", index=False)

else:
    workflow_changes_df = pd.read_parquet(AIDev_v5_pop_path)

print("Commit-file rows:", len(workflow_changes_df))
print("Unique PRs:", workflow_changes_df["pr_id"].nunique())

workflow_changes_df.head()

Commit-file rows: 395260
Unique PRs: 25563


,sha,pr_id,author,committer,message,commit_stats_total,commit_stats_additions,commit_stats_deletions,filename,status,...,repo_url,html_url,id_repo,url,license,full_name,is_forked,language,forks,stars
0,254a57bc6756c16891645e0932544bbe9ff486f2,3071947527,csvoss,csvoss,Switch from Travis CI to GitHub Actions,29,24,5,.github/workflows/ci.yml,added,...,https://api.github.com/repos/csvoss/onelinerizer,https://github.com/csvoss/onelinerizer/pull/87,29375764,https://api.github.com/repos/csvoss/onelinerizer,MIT,csvoss/onelinerizer,False,Python,113,1527
1,045b732a1da0b9e019e84bd2c7546eb573373a85,3071947527,csvoss,web-flow,Update ci.yml,2,1,1,.github/workflows/ci.yml,modified,...,https://api.github.com/repos/csvoss/onelinerizer,https://github.com/csvoss/onelinerizer/pull/87,29375764,https://api.github.com/repos/csvoss/onelinerizer,MIT,csvoss/onelinerizer,False,Python,113,1527
2,c09b2cb677a845cd2029a67992480b3171240f2b,4174019849,riderx,riderx,fix(ci): link supabase templates in tests work...,2,2,0,.github/workflows/tests.yml,modified,...,https://api.github.com/repos/Cap-go/capgo,https://github.com/Cap-go/capgo/pull/1877,442321089,https://api.github.com/repos/Cap-go/capgo,AGPL-3.0,Cap-go/capgo,False,TypeScript,82,187
3,6c491b8602d78f0a6d86c556a77c8320477025b1,3754768299,fjord-oai,fjord-oai,Use PR linux binary in npm staging,44,44,0,.github/workflows/ci.yml,modified,...,https://api.github.com/repos/openai/codex,https://github.com/openai/codex/pull/8440,965415649,https://api.github.com/repos/openai/codex,Apache-2.0,openai/codex,False,Rust,10506,74448
4,4639f3b24a33dbd703af8bc8d07c9bc9af1c3a84,3754768299,fjord-oai,fjord-oai,fix,2,1,1,.github/workflows/ci.yml,modified,...,https://api.github.com/repos/openai/codex,https://github.com/openai/codex/pull/8440,965415649,https://api.github.com/repos/openai/codex,Apache-2.0,openai/codex,False,Rust,10506,74448


In [4]:
# Inspect a single PR

# workflow_changes_df
# workflow_changes_df[workflow_changes_df["pr_id"] == 3071947527]
workflow_changes_df[workflow_changes_df["pr_id"] == 3754768299]

,sha,pr_id,author,committer,message,commit_stats_total,commit_stats_additions,commit_stats_deletions,filename,status,...,repo_url,html_url,id_repo,url,license,full_name,is_forked,language,forks,stars
3,6c491b8602d78f0a6d86c556a77c8320477025b1,3754768299,fjord-oai,fjord-oai,Use PR linux binary in npm staging,44,44,0,.github/workflows/ci.yml,modified,...,https://api.github.com/repos/openai/codex,https://github.com/openai/codex/pull/8440,965415649,https://api.github.com/repos/openai/codex,Apache-2.0,openai/codex,False,Rust,10506,74448
4,4639f3b24a33dbd703af8bc8d07c9bc9af1c3a84,3754768299,fjord-oai,fjord-oai,fix,2,1,1,.github/workflows/ci.yml,modified,...,https://api.github.com/repos/openai/codex,https://github.com/openai/codex/pull/8440,965415649,https://api.github.com/repos/openai/codex,Apache-2.0,openai/codex,False,Rust,10506,74448
5,3b50d08b24f3424b046f2699eef54d4915851b11,3754768299,fjord-oai,fjord-oai,increase timeout,2,1,1,.github/workflows/ci.yml,modified,...,https://api.github.com/repos/openai/codex,https://github.com/openai/codex/pull/8440,965415649,https://api.github.com/repos/openai/codex,Apache-2.0,openai/codex,False,Rust,10506,74448
6,722b4a3b29ae8815137ea85d3688836b92661928,3754768299,fjord-oai,fjord-oai,revert ci changes,17,1,16,.github/workflows/ci.yml,modified,...,https://api.github.com/repos/openai/codex,https://github.com/openai/codex/pull/8440,965415649,https://api.github.com/repos/openai/codex,Apache-2.0,openai/codex,False,Rust,10506,74448


The dataset's `status` describes an **individual commit's** file change. The overall PR can have a different outcome (for example, add then remove the same file).

The recovered pair's `change_type` describes the overall before/after comparison:
- `added`: file absent before; all after findings count as introduced.
- `modified`: file exists on both sides and contents changed.
- `renamed`: old and new workflow paths differ; retain and inspect even if contents match.
- `deleted`: file absent after. Retain separately; disappearing findings do not establish task success.

Do not call these confirmed policy violations without interpreting the rule and context.

## Pair workflows using the background pipeline's method

The earlier cells explore your local AIDev download. The cells below use the **same pinned input tables and pairing functions as the completed background run**. The cohort can differ from your local filter: a refreshed/local dataset is not automatically the pinned snapshot.

One pair means **one workflow file across the whole recorded PR**, not one intermediate commit. Before = first parent of the unique first recorded PR commit. After = unique final recorded PR commit. These are dataset boundaries, not proof of exclusive agent authorship or the PR's eventual final state.

Read the next three stages in order: choose commits; resolve workflow paths; fetch the two snapshots. The network/cache helpers are imported from the production code so we do not maintain a second retrieval implementation. Nothing starts a full run automatically.

In [5]:
# Shared retrieval helpers: same GitHub requests, cache, retries, and absence checks.
import sys
import hashlib
import difflib
import json
from pathlib import Path

workspace = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "Experiments/Policy Oracle Pipeline/paper_a/recover_pairs.py").is_file())
pipeline_dir = workspace / "Experiments/Policy Oracle Pipeline/paper_a"
if str(pipeline_dir) not in sys.path:
    sys.path.insert(0, str(pipeline_dir))

import recover_pairs as background
from assemble_pairs import comparison_qualification

background_outputs = background.OUT
CACHE = background.CACHE
REVISION = background.REVISION
# gh reads authenticated API responses; raw retrieves full immutable file contents.
gh = background.gh
raw = background.raw
parent_path = background.parent_path
public_comparison = background.public_comparison
save_json = background.save_json

# Rebuilt checkpoints go here, never over the background run's recovered PRs.
pairs_dir = Path(r"E:\!My Files\CS\Grammar\PaperA_Notebook_Pairs") / "background_aligned"
RESULTS = pairs_dir / "recovered_prs"

print("Pinned dataset revision:", REVISION)
print("Background data:", background_outputs)
print("Optional rebuild destination:", RESULTS)

Pinned dataset revision: 37bbe1533e26cc1e1374917dba1186d1c8a4dc81
Background data: E:\!My Files\CS\Grammar\WorkflowResearch_Generated\paper_a_outputs
Optional rebuild destination: E:\!My Files\CS\Grammar\PaperA_Notebook_Pairs\background_aligned\recovered_prs


### Inputs: use the exact recorded cohort

`workflow_prs` contains PR metadata, `workflow_pr_commits` contains the frozen PR commit lists, and `workflow_commit_files` contains candidate paths from individual commits. The background selection includes files under `.github/workflows/` before narrowing to YAML pairs, so the PR count is not identical to the earlier YAML-only local filter.

We use these existing small derived tables rather than re-downloading the raw dataset. Reusing inputs is **not an independent validation of how the cohort was originally extracted**.

In [6]:
pinned_prs_df = pd.read_parquet(background_outputs / "workflow_prs.parquet")
pinned_commits_df = pd.read_parquet(background_outputs / "workflow_pr_commits.parquet")
pinned_files_df = pd.read_parquet(background_outputs / "workflow_commit_files.parquet")

pr_info = pinned_prs_df.set_index("id", verify_integrity=True)
commits_by_pr = pinned_commits_df.groupby("pr_id")["sha"].agg(set).to_dict()
files_by_pr = pinned_files_df.groupby("pr_id")["filename"].agg(set).to_dict()

# Same comparison qualification used when the background pairs were assembled.
audit_path = background_outputs / "comparison_audit.json"
audit = json.loads(audit_path.read_text(encoding="utf-8")) if audit_path.exists() else {}
audited_comparisons = {
    (r["pr_id"], r["base_sha"], r["head_sha"])
    for r in audit.get("results", [])
    if r["status"] == "ok" and r.get("workflow_metadata_equal") is True
}
print("Pinned PRs:", len(pr_info))
print("Local YAML-filter PRs (exploration only):", workflow_changes_df["pr_id"].nunique())

Pinned PRs: 25889
Local YAML-filter PRs (exploration only): 25563


### 1. Choose the before and after commits

The function below is the background function, formatted for readability. Parent links determine order; table order and timestamps do not. Missing commits, multiple starting/ending commits, or a first commit without a parent cause an exclusion.

This reproduces the background check exactly. It does **not** add the notebook's previous extra graph traversal check, nor prove that the dataset contains every revision of a PR.

In [7]:
def commit_boundaries(frozen, by_sha):
    if not frozen or frozen - set(by_sha):
        raise ValueError('Pinned commit set empty or not fully recoverable')
    parents = {sha: [p['sha'] for p in by_sha[sha]['parents']] for sha in frozen}
    first = [sha for sha in frozen if not any((p in frozen for p in parents[sha]))]
    referenced = {p for values in parents.values() for p in values if p in frozen}
    heads = sorted(frozen - referenced)
    if len(first) != 1 or len(heads) != 1 or (not parents[first[0]]):
        raise ValueError('Ambiguous PR commit graph or parentless first commit')
    return (parents[first[0]][0], heads[0], first[0])

### 2. Choose workflow paths and handle renames

Start from paths recorded in the dataset. Use the overall comparison to connect old and new names. A workflow renamed within the workflow directory should become one pair, not two. If moved outside it, the background pipeline represents the old workflow path disappearing. Unchanged contents at the same path are skipped later.

In [8]:
def workflow_path(path):
    return path.startswith('.github/workflows/') and path.lower().endswith(('.yml', '.yaml'))

def canonical_paths(files, compare_files):
    """A renamed workflow is one pair, independent of lexical path ordering."""
    candidates = set(files)
    for row in compare_files:
        old = row.get('previous_filename', '')
        if row.get('status') == 'renamed' and (row['filename'] in files or old in files):
            if workflow_path(row['filename']):
                candidates.add(row['filename'])
            elif workflow_path(old):
                candidates.add(old)
    old_to_new = {r['previous_filename']: r['filename'] for r in compare_files if r.get('status') == 'renamed' and r.get('previous_filename')}
    after_paths = {row['filename'] for row in compare_files if row.get('status') != 'removed'}
    return sorted((p for p in candidates if p not in old_to_new or old_to_new[p] not in candidates or (not workflow_path(old_to_new[p])) or (p in after_paths)))

### 3. Recover complete files at both commits

This is the **same recovery function as the background code**, formatted for readability. Its arguments are one PR's metadata, frozen commits, and candidate paths.

1. Find all frozen commits, using cached parents first and GitHub fallbacks if needed.
2. Choose the two snapshot SHAs with `commit_boundaries`.
3. Obtain comparison metadata, using the same saved REST/public-diff routes and fallbacks.
4. Resolve candidate workflow paths and retrieve full before/after contents.
5. Keep file-retrieval failures as excluded pairs; never treat a failed request as an absent file.
6. Skip net-unchanged files; classify the others as added, modified, deleted, or renamed.

**Absence:** the background format stores empty text plus `before_exists=False` or `after_exists=False`. An existing empty file instead has an existence flag of `True`. Always read both fields together. A rename may remain a pair even when its contents are unchanged.

**Qualifications:** a capped REST comparison can leave rename metadata uncertain. The assembly step retains the relevant warning instead of silently treating its metadata as complete. A successful recovery can also contain individual excluded file pairs.

Running this definition does not fetch anything. A fresh recovery call can use the network and populate shared caches; its PR checkpoints are written only to the separate notebook directory.

In [ ]:
def recover(pr, commits, files, force=False):
    path = RESULTS / f"{pr['id']}.json"
    if path.exists() and (not force):
        try:
            cached = json.loads(path.read_text(encoding='utf-8'))
            if cached.get('recovery_revision') == 2:
                return dict(cached, _checkpoint_reused=True)
        except ValueError:
            pass
    result = dict(pr_id=pr['id'], repo=pr['repo'], status='excluded', reason='', pairs=[], recovery_revision=2)
    frozen = set(commits)
    by_sha = {}
    prefetched = []
    for sha in frozen:
        try:
            metadata = json.loads(parent_path(pr['repo'], sha).read_text(encoding='utf-8'))
            if metadata.get('complete') and metadata.get('repo') == pr['repo'] and (metadata.get('sha') == sha):
                by_sha[sha] = dict(sha=sha, parents=metadata['parents'])
                prefetched.append(sha)
        except (OSError, ValueError, KeyError):
            pass
    if frozen - set(by_sha):
        response = gh(f"repos/{pr['repo']}/pulls/{pr['number']}/commits?per_page=100")
        live = [row for page in response['response'] for row in page] if response['status'] == 'ok' else []
        by_sha.update({row['sha']: row for row in live})
    result['commits_from_prefetch'] = prefetched
    recovered_direct = []
    for sha in sorted(frozen - set(by_sha)):
        direct = gh(f"repos/{pr['repo']}/commits/{sha}")
        if direct['status'] == 'ok' and direct['response'][0].get('sha') == sha:
            by_sha[sha] = direct['response'][0]
            recovered_direct.append(sha)
    result['commits_recovered_directly'] = recovered_direct
    try:
        base, head, first = commit_boundaries(frozen, by_sha)
    except ValueError as exc:
        result['reason'] = str(exc)
        result['missing_commits'] = sorted(frozen - set(by_sha))
        save_json(path, result)
        return result
    result.update(base_sha=base, head_sha=head, base_sha_method='first_parent_of_unique_first_pinned_PR_commit', first_pr_commit=first, status='recovered')
    endpoint = f"repos/{pr['repo']}/compare/{base}...{head}"
    comparison_path = CACHE / (hashlib.sha256(endpoint.encode()).hexdigest() + '.json')
    comparison = None
    try:
        cached = json.loads(comparison_path.read_text(encoding='utf-8'))
        if cached['status'] == 'ok' and len(cached['response'][0].get('files', [])) < 300:
            comparison = cached
    except (OSError, ValueError, KeyError):
        pass
    compare_files = []
    if comparison is None:
        public = public_comparison(pr['repo'], base, head)
        if public['status'] == 'ok':
            compare_files = public['files']
            result['comparison_source'] = public['source']
            result['comparison_url'] = public['url']
            result['comparison_sha256'] = public['sha256']
        else:
            comparison = gh(endpoint)
            result['public_comparison_error'] = public.get('error')
    if comparison is not None:
        result['comparison_source'] = 'GitHub_REST_compare'
        if comparison['status'] == 'ok':
            compare_files = comparison['response'][0].get('files', [])
    result['comparison_file_count'] = len(compare_files)
    result['comparison_may_be_truncated'] = result['comparison_source'] == 'GitHub_REST_compare' and len(compare_files) >= 300
    aliases = {x['filename']: x for x in compare_files}
    for meta in compare_files:
        old = meta.get('previous_filename', '')
        if meta.get('status') == 'renamed' and workflow_path(old) and (not workflow_path(meta['filename'])):
            aliases.setdefault(old, dict(filename=old, status='removed'))
    candidates = canonical_paths(files, compare_files)
    for filename in candidates:
        if not workflow_path(filename):
            continue
        meta = aliases.get(filename, {})
        before = meta.get('previous_filename', filename)
        old = raw(pr['repo'], base, before, absence_known=meta.get('status') == 'added')
        new = raw(pr['repo'], head, filename, absence_known=meta.get('status') == 'removed')
        row = dict(pair_id=f"aidev_v5:{pr['repo']}:{pr['id']}:{filename}", source='aidev_v5', source_revision=REVISION, repo=pr['repo'], commit_or_pr=str(pr['id']), pr_number=pr['number'], agent_name=pr['agent'], author_class='agent', stars=pr['stars'], state=pr['state'], created_at=pr['created_at'], closed_at=pr['closed_at'], merged_at=pr['merged_at'], path=filename, before_path=before, after_path=filename, base_sha=base, head_sha=head, base_sha_method=result['base_sha_method'], before_exists=old['status'] == 'found' if old['status'] != 'error' else None, after_exists=new['status'] == 'found' if new['status'] != 'error' else None, before_yaml=old.get('text') if old['status'] != 'absent' else '', after_yaml=new.get('text') if new['status'] != 'absent' else '', conversion_status='ready', error='', prompt=None, prompt_candidates=[], link_quality='none', attribution_scope='pr', pilot=None)
        if 'error' in [old['status'], new['status']]:
            row.update(conversion_status='excluded', error='Immutable file retrieval failed')
        elif old.get('text') == new.get('text') and before == filename:
            continue
        row['change_type'] = 'added' if old['status'] == 'absent' else 'deleted' if new['status'] == 'absent' else 'renamed' if before != filename else 'modified'
        row['github_patch'] = meta.get('patch')
        row['patch_recovery_method'] = result['comparison_source'] if meta.get('patch') else 'full_immutable_GitHub_contents'
        if row['conversion_status'] == 'ready':
            row['complete_patch'] = ''.join(difflib.unified_diff(row['before_yaml'].splitlines(True), row['after_yaml'].splitlines(True), fromfile='a/' + before, tofile='b/' + filename))
        result['pairs'].append(row)
    result['net_unchanged_or_non_yaml_candidates'] = len(candidates) - len(result['pairs'])
    save_json(path, result)
    return result

In [9]:
def build_pr_result(pr_id, rebuild=False):
    """Reuse a recorded result, or explicitly rebuild with the same recovery logic."""
    pr_id = int(pr_id)
    original = background.RESULTS / f"{pr_id}.json"
    if not rebuild:
        # Read only: this is reuse of a completed result, not a fresh reproduction.
        result = json.loads(original.read_text(encoding="utf-8"))
    else:
        pr = pr_info.loc[pr_id].to_dict()
        pr["id"] = pr_id
        result = recover(pr, commits_by_pr.get(pr_id, set()),
                         files_by_pr.get(pr_id, set()), force=True)

    # Same metadata qualification as assemble_pairs.py; do not drop the warning.
    qualification = comparison_qualification(result, audited_comparisons)
    for pair in result["pairs"]:
        pair["metadata_error"] = qualification or pair.get("metadata_error")
    return result

# Start by reading one already recovered PR. Set rebuild=True only to reconstruct it.
example_result = build_pr_result(3071947527, rebuild=False)
example_pairs = example_result["pairs"]
print("Recovery status:", example_result["status"])
print("Exclusion reason:", example_result.get("reason", ""))
if example_pairs:
    display(pd.DataFrame(example_pairs).drop(
        columns=["before_yaml", "after_yaml", "complete_patch", "github_patch"], errors="ignore"))
    pair = example_pairs[0]
    for side in ["before", "after"]:
        state = pair[f"{side}_exists"]
        text = pair[f"{side}_yaml"] if state is True else "[file absent]" if state is False else "[unknown]"
        print(f"\n{side.upper()}:\n", text)

Recovery status: recovered
Exclusion reason: 


,pair_id,source,source_revision,repo,commit_or_pr,pr_number,agent_name,author_class,stars,state,...,conversion_status,error,prompt,prompt_candidates,link_quality,attribution_scope,pilot,change_type,patch_recovery_method,metadata_error
0,aidev_v5:csvoss/onelinerizer:3071947527:.githu...,aidev_v5,37bbe1533e26cc1e1374917dba1186d1c8a4dc81,csvoss/onelinerizer,3071947527,87,OpenAI_Codex,agent,1527,open,...,ready,,None,[],none,pr,None,added,github_compare,None



BEFORE:
 [file absent]

AFTER:
 name: CI

on:
  push:
    branches: [ master ]
  pull_request:

jobs:
  build:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11"]
    steps:
    - uses: actions/checkout@v3
    - uses: actions/setup-python@v4
      with:
        python-version: ${{ matrix.python-version }}
    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip setuptools
    - name: Run tests
      run: python setup.py test



### Optional small rebuild and comparison

Leave `RUN_REBUILD=False` while reading. Turning it on rebuilds only the first **five** selected PRs, using the same recovery logic. Existing immutable caches may avoid requests, but missing information can trigger GitHub calls. The background cache can be populated; original recovered results and raw datasets are not overwritten.

The comparison checks actual SHAs, paths, existence flags, change types, contents, and qualification fields against saved results. It is an implementation/reproduction check over this small sample—not independent validation of GitHub history or proof of full-cohort agreement. Old notebook `.json.gz` checkpoints are not used because their method differed.

In [ ]:
RUN_REBUILD = False
PR_LIMIT = 5

if RUN_REBUILD:
    comparison_rows = []
    fields = ["pair_id", "base_sha", "head_sha", "before_path", "after_path",
              "before_exists", "after_exists", "change_type", "before_yaml", "after_yaml",
              "conversion_status", "metadata_error"]
    for pr_id in sorted(pr_info.index)[:PR_LIMIT]:
        try:
            expected = build_pr_result(pr_id, rebuild=False)
            rebuilt = build_pr_result(pr_id, rebuild=True)
            def keyed(result):
                return {p["pair_id"]: {field: p.get(field) for field in fields}
                        for p in result["pairs"]}
            comparison_rows.append({
                "pr_id": int(pr_id), "status": rebuilt["status"],
                "same_status": expected["status"] == rebuilt["status"],
                "same_pairs": keyed(expected) == keyed(rebuilt), "error": None,
            })
        except Exception as error:
            comparison_rows.append({"pr_id": int(pr_id), "status": "error",
                                    "same_status": False, "same_pairs": False, "error": str(error)})
    comparison_df = pd.DataFrame(comparison_rows)
    pairs_dir.mkdir(parents=True, exist_ok=True)
    comparison_df.to_csv(pairs_dir / "rebuild_comparison.csv", index=False)
    display(comparison_df)
else:
    print("Rebuild disabled. No recovery requests or new PR checkpoints created.")

### What was checked when aligning this notebook

The four visible functions (`commit_boundaries`, `workflow_path`, `canonical_paths`, `recover`) were compared structurally against `Experiments/Policy Oracle Pipeline/paper_a/recover_pairs.py`: their executable logic matched exactly. This is a readable copy, so later edits to the production file will not automatically update it.

Three PRs were reconstructed using cached data only: `3071947527`, `2756921963`, and `2759620798`. All four resulting workflow pairs matched the saved PR recovery results for snapshot SHAs, paths, existence flags, change types, full contents, and conversion status. This covered added, modified, and deleted files; it did not establish rename or full-cohort reproduction. Temporary validation checkpoints were discarded. No new network requests or scanner runs were made.

The setup, pinned-input loading, recorded preview, and disabled-rebuild cells were also exercised outside the notebook. Outputs below are cleared deliberately: restart the kernel and run the cells yourself to obtain your own outputs. The earlier raw-data exploration cells were preserved and were not rerun as part of this check.


## Continue with the completed background pairs

The next cell reads the existing final AIDev pairs for the scanner walkthrough. This is the published pairing output, not output from the optional notebook rebuild. Its file-level rows and existence flags use the same method above. The final scan cell still runs zizmor on one example; finding matching remains a later step.

In [ ]:
# Preview AIDev pairs already recovered by the background pipeline.
background_pairs_path = r"E:\!My Files\CS\Grammar\WorkflowResearch_Generated\paper_a_outputs\aidev_pairs.parquet"
background_pairs_file = pq.ParquetFile(background_pairs_path)
print("Saved AIDev workflow pairs:", background_pairs_file.metadata.num_rows)

preview_columns = [
    "repo", "commit_or_pr", "agent_name", "path", "before_path", "after_path", "change_type",
    "base_sha", "head_sha", "before_exists", "after_exists",
    "before_yaml", "after_yaml",
]
background_pairs_preview_df = next(
    background_pairs_file.iter_batches(batch_size=5, columns=preview_columns)
).to_pandas()

display(background_pairs_preview_df.drop(columns=["before_yaml", "after_yaml"]))

# Inspect the first pair. Full contents remain in background_pairs_preview_df.
background_pair_row = background_pairs_preview_df.iloc[0]
print("\nBEFORE (first 2,000 characters):\n",
      background_pair_row["before_yaml"][:2000] if background_pair_row["before_exists"] else "[file absent]")
print("\nAFTER (first 2,000 characters):\n",
      background_pair_row["after_yaml"][:2000] if background_pair_row["after_exists"] else "[file absent]")


Saved AIDev workflow pairs: 241504


,repo,commit_or_pr,agent_name,path,before_path,after_path,change_type,base_sha,head_sha,before_exists,after_exists
0,AgentOps-AI/agentops,2756921963,Devin,.github/workflows/python-testing.yml,.github/workflows/python-testing.yml,.github/workflows/python-testing.yml,modified,915cb590af512bad09f0a238458a40647e2124a8,fc2438ec3b15d03dcf3e31e52e4ad7eb251cc033,True,True
1,safety-research/safety-tooling,2758858933,Devin,.github/workflows/lint_and_test.yml,.github/workflows/lint_and_test.yml,.github/workflows/lint_and_test.yml,added,f74da1a78f3b9742fea5dfbf11f0f78c2dde3d2e,4e7939d16151da1df2640f757c595215180dc4a3,False,True
2,Nexus-JPF/note-companion,2759596477,Devin,.github/workflows/manual-release.yml,.github/workflows/manual-release.yml,.github/workflows/manual-release.yml,added,79d670e3bb5658a58221facbb0baf4c0b8082477,b2c79c0e893e1069a9bcf6eec7644fe5fb1a6344,False,True
3,Nexus-JPF/note-companion,2759620798,Devin,.github/workflows/manual-release.yml,.github/workflows/manual-release.yml,.github/workflows/manual-release.yml,modified,c95cfef50485ab79a47ba0e7f08ce55d0ca1fd7b,2f9650232a1c31fcf09e5e67dafe28d5db368d89,True,True
4,Nexus-JPF/note-companion,2759620798,Devin,.github/workflows/release.yml,.github/workflows/release.yml,.github/workflows/release.yml,deleted,c95cfef50485ab79a47ba0e7f08ce55d0ca1fd7b,2f9650232a1c31fcf09e5e67dafe28d5db368d89,True,False



BEFORE (first 2,000 characters):
 name: Python Tests

on:
  workflow_dispatch:
  push:
    branches:
      - main
    paths:
      - 'agentops/**/*.py'
      - 'agentops/**/*.ipynb'
      - 'tests/**/*.py'
      - 'tests/**/*.ipynb'
  pull_request:
    branches:
      - main
    paths:
      - 'agentops/**/*.py'
      - 'agentops/**/*.ipynb'
      - 'tests/**/*.py'
      - 'tests/**/*.ipynb'

jobs:
  build:
    runs-on: ubuntu-latest
    env:
      OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}

    strategy:
      matrix:
        python-version: ["3.7","3.8","3.9","3.10","3.11","3.12"]

    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          cache: 'pip'
          python-version: '3.11'
      - name: Install tox
        run: pip install tox
      - name: Run tests with tox
        run: tox
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
          AGENTOPS_API_KEY: ${{ secrets.AGENTOPS_API_KEY }}


AFTER (first 2,0

Run zizmor on one saved pair. This cell shows raw before/after findings and records each side's scan status. It uses the same two audit exclusions as the background run. Strict collection makes malformed inputs an error. Finding matching comes next; these rows do not yet have introduced/fixed/persisted labels. Rerun the preview cell first to load both file paths for renamed workflows.


In [ ]:
# Run zizmor on one saved workflow pair, before and after.
zizmor_exe = Path(r"C:\Users\mekae\Desktop\CS\Masters\Workflow Research\Experiments\Policy Oracle Pipeline\paper_a\tools\zizmor.exe")
# Same audit exclusions as the background run; remove IDs here to include them.
excluded_audits = {"self-repository", "archived-uses"}


def run_zizmor(yaml_text, workflow_path):
    if not isinstance(yaml_text, str):
        raise ValueError("Workflow content is missing")
    with tempfile.TemporaryDirectory() as directory:
        target = Path(directory) / workflow_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(yaml_text, encoding="utf-8", newline="")
        result = subprocess.run(
            [str(zizmor_exe), "--offline", "--format", "json",
             "--min-severity", "low", "--strict-collection", "--no-exit-codes", str(target)],
            capture_output=True, text=True, encoding="utf-8", timeout=60,
            cwd=directory,
        )
    if result.returncode:
        raise RuntimeError(result.stderr.strip())
    findings = json.loads(result.stdout)
    if not isinstance(findings, list):
        raise ValueError("Unexpected scanner output")
    return [f for f in findings if not f.get("ignored") and f["ident"] not in excluded_audits]


# Change this index to inspect another of the five previewed pairs.
scan_pair = background_pairs_preview_df.iloc[0]
zizmor_example_result = {
    "repo": scan_pair["repo"], "pr_id": scan_pair["commit_or_pr"],
    "path": scan_pair["path"], "base_sha": scan_pair["base_sha"],
    "head_sha": scan_pair["head_sha"], "oracle": "zizmor", "version": "1.30.1",
}
side_rows, finding_rows = [], []
for side in ["before", "after"]:
    try:
        exists = scan_pair[f"{side}_exists"]
        if pd.isna(exists):
            raise ValueError("File existence is unknown")
        path = scan_pair.get(f"{side}_path", scan_pair["path"])
        active = path.startswith(".github/workflows/") and path.lower().endswith((".yml", ".yaml"))
        if not exists:
            outcome = {"status": "file_absent", "findings": [], "error": None}
        elif not active:
            outcome = {"status": "not_active_workflow", "findings": [], "error": None}
        else:
            findings = run_zizmor(scan_pair[f"{side}_yaml"], path)
            outcome = {"status": "ok", "findings": findings, "error": None}
    except Exception as error:
        outcome = {"status": "error", "findings": [], "error": str(error)}

    zizmor_example_result[side] = outcome
    side_rows.append({"side": side, "status": outcome["status"],
                      "n_findings": None if outcome["status"] == "error" else len(outcome["findings"]),
                      "error": outcome["error"]})
    for finding in outcome["findings"]:
        finding_rows.append({"side": side, "rule_id": finding["ident"],
                             "severity": finding.get("determinations", {}).get("severity"),
                             "description": finding.get("desc")})

zizmor_side_summary_df = pd.DataFrame(side_rows)
zizmor_findings_df = pd.DataFrame(finding_rows, columns=["side", "rule_id", "severity", "description"])
display(zizmor_side_summary_df)
display(zizmor_findings_df)

scanner_output_path = Path(r"E:\!My Files\CS\Grammar\PaperA_Notebook_Pairs") / "zizmor_example.json"
scanner_output_path.parent.mkdir(parents=True, exist_ok=True)
scanner_output_path.write_text(json.dumps(zizmor_example_result, indent=2), encoding="utf-8")
print("Saved raw findings and scan statuses to:", scanner_output_path)


## Explanation review: browse the completed dataset

Run these small cells in order. They read existing results; they do not run scanners or make network requests. Full YAML stays on disk until you select a case.

One row in `review_cases_df` represents **one workflow-PR pair, checker, rule, and finding status**. `instances` counts its underlying flags. Paths can recur across PRs and rules. These are candidate cases, not manually confirmed violations or a representative random sample.

### 1. Imports and results folder

In [44]:
import json
import difflib
from pathlib import Path
from urllib.parse import quote
import pandas as pd
from IPython.display import display

results_dir = Path(r"E:\!My Files\CS\Grammar\WorkflowResearch_Generated\paper_a_outputs")

### 2. Load the recovered workflow-pair metadata

Keep only ready AIDev pairs. This does not load their YAML text.

In [45]:
# Metadata only: full workflow texts stay on disk until you open one case.
case_pairs_df = pd.read_parquet(results_dir / "full_run/pairs_summary.parquet", columns=[
    "pair_id", "source", "repo", "commit_or_pr", "agent_name", "path",
    "change_type", "conversion_status", "scan_status"])
case_pairs_df = case_pairs_df.query("source == 'aidev_v5' and conversion_status == 'ready'").copy()

In [46]:
case_pairs_df

,pair_id,source,repo,commit_or_pr,agent_name,path,change_type,conversion_status,scan_status
0,aidev_v5:ESPresense/ESPresense:3614012357:.git...,aidev_v5,ESPresense/ESPresense,3614012357,OpenAI_Codex,.github/workflows/qemu-tests.yml,added,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ..."
1,aidev_v5:rmusser01/tldw_server:3976895418:.git...,aidev_v5,rmusser01/tldw_server,3976895418,OpenAI_Codex,.github/workflows/actionlint.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ..."
2,aidev_v5:rmusser01/tldw_server:3976895418:.git...,aidev_v5,rmusser01/tldw_server,3976895418,OpenAI_Codex,.github/workflows/codeql.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ..."
3,aidev_v5:rmusser01/tldw_server:3976895418:.git...,aidev_v5,rmusser01/tldw_server,3976895418,OpenAI_Codex,.github/workflows/frontend-ux-gates.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ..."
4,aidev_v5:rmusser01/tldw_server:3976895418:.git...,aidev_v5,rmusser01/tldw_server,3976895418,OpenAI_Codex,.github/workflows/sbom.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ..."
...,...,...,...,...,...,...,...,...,...
248105,aidev_v5:github/gh-aw:4104395262:.github/workf...,aidev_v5,github/gh-aw,4104395262,Copilot,.github/workflows/workflow-generator.lock.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ..."
248106,aidev_v5:github/gh-aw:4104395262:.github/workf...,aidev_v5,github/gh-aw,4104395262,Copilot,.github/workflows/workflow-health-manager.lock...,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ..."
248107,aidev_v5:github/gh-aw:4104395262:.github/workf...,aidev_v5,github/gh-aw,4104395262,Copilot,.github/workflows/workflow-normalizer.lock.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ..."
248108,aidev_v5:github/gh-aw:4104395262:.github/workf...,aidev_v5,github/gh-aw,4104395262,Copilot,.github/workflows/workflow-skill-extractor.loc...,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ..."


### 3. Attach snapshot paths and PR numbers

Join on the unique `pair_id`, keeping retrieval/metadata warnings.

In [47]:
case_metadata_df = pd.read_parquet(results_dir / "pairs.parquet", columns=[
    "pair_id", "pr_number", "base_sha", "head_sha", "before_path", "after_path",
    "before_exists", "after_exists", "metadata_error"])
case_pairs_df = case_pairs_df.merge(case_metadata_df, on="pair_id", validate="one_to_one")

In [48]:
case_pairs_df

,pair_id,source,repo,commit_or_pr,agent_name,path,change_type,conversion_status,scan_status,pr_number,base_sha,head_sha,before_path,after_path,before_exists,after_exists,metadata_error
0,aidev_v5:ESPresense/ESPresense:3614012357:.git...,aidev_v5,ESPresense/ESPresense,3614012357,OpenAI_Codex,.github/workflows/qemu-tests.yml,added,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",1888.0,22e7add84eb1b3a37e6ef2f0afd0867c31e7c7c8,a4d9a8a47cf0d56d66651a3c9308a9e8e14c1684,.github/workflows/qemu-tests.yml,.github/workflows/qemu-tests.yml,False,True,None
1,aidev_v5:rmusser01/tldw_server:3976895418:.git...,aidev_v5,rmusser01/tldw_server,3976895418,OpenAI_Codex,.github/workflows/actionlint.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",762.0,ad6b30bbcae97f2d23fafef226b7b8d3926d041b,8834d4d464f46c856a4e02229113213f5bbe3f04,.github/workflows/actionlint.yml,.github/workflows/actionlint.yml,True,True,None
2,aidev_v5:rmusser01/tldw_server:3976895418:.git...,aidev_v5,rmusser01/tldw_server,3976895418,OpenAI_Codex,.github/workflows/codeql.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",762.0,ad6b30bbcae97f2d23fafef226b7b8d3926d041b,8834d4d464f46c856a4e02229113213f5bbe3f04,.github/workflows/codeql.yml,.github/workflows/codeql.yml,True,True,None
3,aidev_v5:rmusser01/tldw_server:3976895418:.git...,aidev_v5,rmusser01/tldw_server,3976895418,OpenAI_Codex,.github/workflows/frontend-ux-gates.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",762.0,ad6b30bbcae97f2d23fafef226b7b8d3926d041b,8834d4d464f46c856a4e02229113213f5bbe3f04,.github/workflows/frontend-ux-gates.yml,.github/workflows/frontend-ux-gates.yml,True,True,None
4,aidev_v5:rmusser01/tldw_server:3976895418:.git...,aidev_v5,rmusser01/tldw_server,3976895418,OpenAI_Codex,.github/workflows/sbom.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",762.0,ad6b30bbcae97f2d23fafef226b7b8d3926d041b,8834d4d464f46c856a4e02229113213f5bbe3f04,.github/workflows/sbom.yml,.github/workflows/sbom.yml,True,True,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241499,aidev_v5:github/gh-aw:4104395262:.github/workf...,aidev_v5,github/gh-aw,4104395262,Copilot,.github/workflows/workflow-generator.lock.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",21859.0,759d1461747a3fa7e7f2a5b6c86ae96ee2f84667,76df0ae35a42a20a25067fd46da74f8ba2c054f5,.github/workflows/workflow-generator.lock.yml,.github/workflows/workflow-generator.lock.yml,True,True,None
241500,aidev_v5:github/gh-aw:4104395262:.github/workf...,aidev_v5,github/gh-aw,4104395262,Copilot,.github/workflows/workflow-health-manager.lock...,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",21859.0,759d1461747a3fa7e7f2a5b6c86ae96ee2f84667,76df0ae35a42a20a25067fd46da74f8ba2c054f5,.github/workflows/workflow-health-manager.lock...,.github/workflows/workflow-health-manager.lock...,True,True,None
241501,aidev_v5:github/gh-aw:4104395262:.github/workf...,aidev_v5,github/gh-aw,4104395262,Copilot,.github/workflows/workflow-normalizer.lock.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",21859.0,759d1461747a3fa7e7f2a5b6c86ae96ee2f84667,76df0ae35a42a20a25067fd46da74f8ba2c054f5,.github/workflows/workflow-normalizer.lock.yml,.github/workflows/workflow-normalizer.lock.yml,True,True,None
241502,aidev_v5:github/gh-aw:4104395262:.github/workf...,aidev_v5,github/gh-aw,4104395262,Copilot,.github/workflows/workflow-skill-extractor.loc...,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",21859.0,759d1461747a3fa7e7f2a5b6c86ae96ee2f84667,76df0ae35a42a20a25067fd46da74f8ba2c054f5,.github/workflows/workflow-skill-extractor.loc...,.github/workflows/workflow-skill-extractor.loc...,True,True,None


### 4. Attach the recorded merge status

Use the pinned dataset metadata, not live GitHub status. A merged label does not prove the scanned snapshot was deployed.

In [49]:
case_prs_df = pd.read_parquet(results_dir / "workflow_prs.parquet", columns=["id", "state", "merged_at"])
case_prs_df["commit_or_pr"] = case_prs_df["id"].astype(str)
case_prs_df["merge_status"] = "unknown"
case_prs_df.loc[case_prs_df.state.eq("open"), "merge_status"] = "open"
case_prs_df.loc[case_prs_df.state.eq("closed"), "merge_status"] = "closed_unmerged"
case_prs_df.loc[case_prs_df.merged_at.notna(), "merge_status"] = "merged"
# Replace any previous merge-status columns so this cell is safe to rerun.
# Also removes _x/_y suffixes left by an earlier repeated merge.
old_status_columns = [
    column for column in case_pairs_df.columns
    if column == "merge_status" or column.startswith("merge_status_")
]
case_pairs_df = case_pairs_df.drop(columns=old_status_columns)
case_pairs_df = case_pairs_df.merge(case_prs_df[["commit_or_pr", "merge_status"]],
                                     on="commit_or_pr", how="left", validate="many_to_one")
assert case_pairs_df.pair_id.is_unique and case_pairs_df.merge_status.notna().all()


In [50]:
case_pairs_df

,pair_id,source,repo,commit_or_pr,agent_name,path,change_type,conversion_status,scan_status,pr_number,base_sha,head_sha,before_path,after_path,before_exists,after_exists,metadata_error,merge_status
0,aidev_v5:ESPresense/ESPresense:3614012357:.git...,aidev_v5,ESPresense/ESPresense,3614012357,OpenAI_Codex,.github/workflows/qemu-tests.yml,added,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",1888.0,22e7add84eb1b3a37e6ef2f0afd0867c31e7c7c8,a4d9a8a47cf0d56d66651a3c9308a9e8e14c1684,.github/workflows/qemu-tests.yml,.github/workflows/qemu-tests.yml,False,True,None,open
1,aidev_v5:rmusser01/tldw_server:3976895418:.git...,aidev_v5,rmusser01/tldw_server,3976895418,OpenAI_Codex,.github/workflows/actionlint.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",762.0,ad6b30bbcae97f2d23fafef226b7b8d3926d041b,8834d4d464f46c856a4e02229113213f5bbe3f04,.github/workflows/actionlint.yml,.github/workflows/actionlint.yml,True,True,None,merged
2,aidev_v5:rmusser01/tldw_server:3976895418:.git...,aidev_v5,rmusser01/tldw_server,3976895418,OpenAI_Codex,.github/workflows/codeql.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",762.0,ad6b30bbcae97f2d23fafef226b7b8d3926d041b,8834d4d464f46c856a4e02229113213f5bbe3f04,.github/workflows/codeql.yml,.github/workflows/codeql.yml,True,True,None,merged
3,aidev_v5:rmusser01/tldw_server:3976895418:.git...,aidev_v5,rmusser01/tldw_server,3976895418,OpenAI_Codex,.github/workflows/frontend-ux-gates.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",762.0,ad6b30bbcae97f2d23fafef226b7b8d3926d041b,8834d4d464f46c856a4e02229113213f5bbe3f04,.github/workflows/frontend-ux-gates.yml,.github/workflows/frontend-ux-gates.yml,True,True,None,merged
4,aidev_v5:rmusser01/tldw_server:3976895418:.git...,aidev_v5,rmusser01/tldw_server,3976895418,OpenAI_Codex,.github/workflows/sbom.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",762.0,ad6b30bbcae97f2d23fafef226b7b8d3926d041b,8834d4d464f46c856a4e02229113213f5bbe3f04,.github/workflows/sbom.yml,.github/workflows/sbom.yml,True,True,None,merged
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241499,aidev_v5:github/gh-aw:4104395262:.github/workf...,aidev_v5,github/gh-aw,4104395262,Copilot,.github/workflows/workflow-generator.lock.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",21859.0,759d1461747a3fa7e7f2a5b6c86ae96ee2f84667,76df0ae35a42a20a25067fd46da74f8ba2c054f5,.github/workflows/workflow-generator.lock.yml,.github/workflows/workflow-generator.lock.yml,True,True,None,closed_unmerged
241500,aidev_v5:github/gh-aw:4104395262:.github/workf...,aidev_v5,github/gh-aw,4104395262,Copilot,.github/workflows/workflow-health-manager.lock...,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",21859.0,759d1461747a3fa7e7f2a5b6c86ae96ee2f84667,76df0ae35a42a20a25067fd46da74f8ba2c054f5,.github/workflows/workflow-health-manager.lock...,.github/workflows/workflow-health-manager.lock...,True,True,None,closed_unmerged
241501,aidev_v5:github/gh-aw:4104395262:.github/workf...,aidev_v5,github/gh-aw,4104395262,Copilot,.github/workflows/workflow-normalizer.lock.yml,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",21859.0,759d1461747a3fa7e7f2a5b6c86ae96ee2f84667,76df0ae35a42a20a25067fd46da74f8ba2c054f5,.github/workflows/workflow-normalizer.lock.yml,.github/workflows/workflow-normalizer.lock.yml,True,True,None,closed_unmerged
241502,aidev_v5:github/gh-aw:4104395262:.github/workf...,aidev_v5,github/gh-aw,4104395262,Copilot,.github/workflows/workflow-skill-extractor.loc...,modified,ready,"{""zizmor_before"": ""analyzed"", ""zizmor_after"": ...",21859.0,759d1461747a3fa7e7f2a5b6c86ae96ee2f84667,76df0ae35a42a20a25067fd46da74f8ba2c054f5,.github/workflows/workflow-skill-extractor.loc...,.github/workflows/workflow-skill-extractor.loc...,True,True,None,closed_unmerged


### 5. Load scanner findings

These are the existing introduced, fixed, and persisted finding instances.

In [51]:
case_findings_df = pd.read_parquet(results_dir / "full_run/findings.parquet", columns=[
    "pair_id", "oracle", "rule_id", "status", "severity", "detail", "anchor",
    "line", "evidence", "before_evidence", "needs_review", "match_basis"],
    filters=[("source", "==", "aidev_v5")])


In [52]:
case_findings_df

,pair_id,oracle,rule_id,status,severity,detail,anchor,line,evidence,before_evidence,needs_review,match_basis
0,aidev_v5:ESPresense/ESPresense:3614012357:.git...,zizmor,Z_artipacked,introduced,Medium,credential persistence through GitHub Actions ...,jobs.qemu.steps[uses:actions/checkout],19.0,uses: actions/checkout@v5,None,False,unmatched
1,aidev_v5:ESPresense/ESPresense:3614012357:.git...,zizmor,Z_excessive_permissions,introduced,Medium,overly broad permissions,jobs.qemu,16.0,qemu: runs-on: ubuntu-22.04 steps: ...,None,False,unmatched
2,aidev_v5:ESPresense/ESPresense:3614012357:.git...,zizmor,Z_unpinned_uses,introduced,High,unpinned action reference,jobs.qemu.steps[uses:actions/checkout],19.0,actions/checkout@v5,None,False,unmatched
3,aidev_v5:ESPresense/ESPresense:3614012357:.git...,zizmor,Z_unpinned_uses,introduced,High,unpinned action reference,jobs.qemu.steps[name:Set up Python],22.0,actions/setup-python@v6,None,False,unmatched
4,aidev_v5:ESPresense/ESPresense:3614012357:.git...,zizmor,Z_unpinned_uses,introduced,High,unpinned action reference,jobs.qemu.steps[name:Run QEMU tests],46.0,tobozo/esp32-qemu-sim@v2.1.0,None,False,unmatched
...,...,...,...,...,...,...,...,...,...,...,...,...
1834112,aidev_v5:github/gh-aw:4104395262:.github/workf...,custom,R9_job_timeouts,persisted,review,Literal candidate requirement not met; applica...,workflow,NaN,R9_job_timeouts,R9_job_timeouts,False,anchor
1834113,aidev_v5:github/gh-aw:4104395262:.github/workf...,custom,R11_no_continue_on_error,persisted,review,Literal candidate requirement not met; applica...,workflow,NaN,R11_no_continue_on_error,R11_no_continue_on_error,False,anchor
1834114,aidev_v5:Z3Prover/z3:4104398985:.github/workfl...,zizmor,Z_excessive_permissions,persisted,Medium,overly broad permissions,jobs.agent,229.0,permissions: read-all,permissions: read-all,False,anchor
1834115,aidev_v5:Z3Prover/z3:4104398985:.github/workfl...,custom,R9_job_timeouts,persisted,review,Literal candidate requirement not met; applica...,workflow,NaN,R9_job_timeouts,R9_job_timeouts,False,anchor


For each case, record: **Is the flag/match correct? What changed? Does the requirement apply? What recurring pattern is visible? What context is missing?** Findings alone do not establish agent intent, policy exposure, or task failure.

### LLM Explanation Generation

In [53]:
# Match finding instances to the eligible workflow pairs.
sampling_df = case_findings_df.merge(
    case_pairs_df,
    on="pair_id",
    how="inner",
    validate="many_to_one",
)

# Fixed and persisted findings will be handled separately.
sampling_df = sampling_df[sampling_df["status"] == "introduced"].copy()

# Keep only findings from successful before-and-after analysis by this checker.
analyzed = [
    json.loads(scan_status).get(oracle) == "analyzed"
    for scan_status, oracle in zip(sampling_df["scan_status"], sampling_df["oracle"])
]
sampling_df = sampling_df.loc[analyzed].copy()


In [54]:
# One case = one workflow pair x scanner x rule.
case_keys = ["pair_id", "oracle", "rule_id"]

# Severity is compared WITHIN a scanner/rule, not across scanners.
# Zizmor: High > Medium > Low. Actionlint keeps "error"; custom/change keep "review".
severity_rank = {"unknown": 0, "Low": 1, "Medium": 2, "High": 3,
                 "error": 1, "review": 1}
assert set(sampling_df["severity"].dropna()).issubset(severity_rank)

case_severities_df = sampling_df.groupby(case_keys).agg(
    severity=("severity", lambda values: max(
        values.dropna(), key=severity_rank.get, default="unknown")),
    all_severities=("severity", lambda values: ", ".join(
        sorted(set(values.dropna().astype(str)))) or "unknown"),
    finding_instances=("rule_id", "size"),
).reset_index()

# Replace the representative row's severity with the MAXIMUM severity of the case.
reduced_sampling_df = (
    sampling_df.groupby(case_keys, group_keys=False)
    .sample(n=1, random_state=10)
    .drop(columns="severity")
    .merge(case_severities_df, on=case_keys, validate="one_to_one")
)

# "severity" below means maximum finding severity per case.
# For LLM input, retrieve ALL original instances for the case from sampling_df;
# detail/evidence/line above still describe only one representative finding.
target_categories = [
    "severity",
    "agent_name",
    "repo",
    "change_type",
    "merge_status",
]


In [55]:
reduced_sampling_df

,pair_id,oracle,rule_id,status,detail,anchor,line,evidence,before_evidence,needs_review,...,head_sha,before_path,after_path,before_exists,after_exists,metadata_error,merge_status,severity,all_severities,finding_instances
0,aidev_v5:0x80/isolate-package:3226855748:.gith...,custom,R1_permissions_declared,introduced,Literal candidate requirement not met; applica...,workflow,NaN,R1_permissions_declared,None,False,...,1ccf6e5f4933668a0b2a4ae636102ca4f13b4007,.github/workflows/checks.yml,.github/workflows/checks.yml,False,True,None,merged,review,review,1
1,aidev_v5:0x80/isolate-package:3226855748:.gith...,custom,R7_all_actions_sha_pinned,introduced,Literal candidate requirement not met; applica...,workflow,NaN,R7_all_actions_sha_pinned,None,False,...,1ccf6e5f4933668a0b2a4ae636102ca4f13b4007,.github/workflows/checks.yml,.github/workflows/checks.yml,False,True,None,merged,review,review,1
2,aidev_v5:0x80/isolate-package:3226855748:.gith...,custom,R8_checkout_no_persist_creds,introduced,Literal candidate requirement not met; applica...,workflow,NaN,R8_checkout_no_persist_creds,None,False,...,1ccf6e5f4933668a0b2a4ae636102ca4f13b4007,.github/workflows/checks.yml,.github/workflows/checks.yml,False,True,None,merged,review,review,1
3,aidev_v5:0x80/isolate-package:3226855748:.gith...,custom,R9_job_timeouts,introduced,Literal candidate requirement not met; applica...,workflow,NaN,R9_job_timeouts,None,False,...,1ccf6e5f4933668a0b2a4ae636102ca4f13b4007,.github/workflows/checks.yml,.github/workflows/checks.yml,False,True,None,merged,review,review,1
4,aidev_v5:0x80/isolate-package:3226855748:.gith...,zizmor,Z_artipacked,introduced,credential persistence through GitHub Actions ...,jobs.verify.steps[uses:actions/checkout],22.0,uses: actions/checkout@v4,None,False,...,1ccf6e5f4933668a0b2a4ae636102ca4f13b4007,.github/workflows/checks.yml,.github/workflows/checks.yml,False,True,None,merged,Medium,Medium,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111136,aidev_v5:zzuu666/antue:3999726102:.github/work...,zizmor,Z_excessive_permissions,introduced,overly broad permissions,workflow,10.0,id-token: write,None,False,...,d784638fca89c1883364c7180680a4596744e3d9,.github/workflows/deploy.yml,.github/workflows/deploy.yml,False,True,None,merged,High,High,2
111137,aidev_v5:zzuu666/antue:3999726102:.github/work...,zizmor,Z_unpinned_uses,introduced,unpinned action reference,jobs.deploy.steps[id:deployment],53.0,actions/deploy-pages@v4,None,False,...,d784638fca89c1883364c7180680a4596744e3d9,.github/workflows/deploy.yml,.github/workflows/deploy.yml,False,True,None,merged,High,High,4
111138,aidev_v5:zzuu666/antue:4001037352:.github/work...,change,D_permissions_broadened,introduced,1 -> 2,jobs.deploy.permissions.contents,NaN,contents,None,True,...,bf8af078c70ddd6027b0a5f605e4a115aefa482c,.github/workflows/deploy.yml,.github/workflows/deploy.yml,True,True,None,merged,review,review,1
111139,aidev_v5:zzuu666/antue:4001037352:.github/work...,custom,R6_third_party_sha_pinned,introduced,Literal candidate requirement not met; applica...,workflow,NaN,R6_third_party_sha_pinned,None,True,...,bf8af078c70ddd6027b0a5f605e4a115aefa482c,.github/workflows/deploy.yml,.github/workflows/deploy.yml,True,True,None,merged,review,review,1


In [56]:
# Target budget: 384 cases per scanner x rule x target DIMENSION.
# Balance across available target values using collection rounds.
# If the rule has fewer than 384 cases, take all.
# This design does not automatically provide Cochran's usual precision.
sample_size = 384
samples = []

for _, rule_group in reduced_sampling_df.groupby(["oracle", "rule_id"]):
    for target in target_categories:
        # Shuffle all cases without replacement.
        shuffled = rule_group.sample(frac=1, random_state=42).copy()

        # First case per value gets round 0, second gets round 1, etc.
        shuffled["_round"] = shuffled.groupby(
            target, dropna=False, observed=True
        ).cumcount()

        selected = (
            shuffled.sort_values("_round", kind="stable")
            .head(sample_size)
            .drop(columns="_round")
        )

        selected["sampling_target"] = target
        samples.append(selected)

# Preserve all memberships: a case may be selected for multiple dimensions.
targeted_sampling_df = pd.concat(samples, ignore_index=True)
targeted_sampling_df


,pair_id,oracle,rule_id,status,detail,anchor,line,evidence,before_evidence,needs_review,...,before_path,after_path,before_exists,after_exists,metadata_error,merge_status,severity,all_severities,finding_instances,sampling_target
0,aidev_v5:Azure/azure-sdk-for-python:3500258934...,actionlint,A_action,introduced,"the runner of ""actions/checkout@v2"" action is ...",jobs.verify-azpysdk-checks.steps[uses:actions/...,59.0,"the runner of ""actions/checkout@v2"" action is ...",None,True,...,.github/workflows/azure-sdk-tools.yml,.github/workflows/azure-sdk-tools.yml,True,True,None,open,error,error,2,severity
1,aidev_v5:eclipse-rdf4j/rdf4j:3462373328:.githu...,actionlint,A_action,introduced,"the runner of ""actions/setup-node@v3"" action i...",jobs.copilot-setup.steps[name:Install Node.js ...,27.0,"the runner of ""actions/setup-node@v3"" action i...",None,False,...,.github/workflows/copilot-setup-steps.yml,.github/workflows/copilot-setup-steps.yml,False,True,None,closed_unmerged,error,error,1,severity
2,aidev_v5:github/gh-aw:3757286500:.github/workf...,actionlint,A_action,introduced,"specifying action ""actions/setup"" in invalid f...",jobs.safe_outputs.steps[name:Setup Scripts]#35...,6212.0,"specifying action ""actions/setup"" in invalid f...",None,True,...,.github/workflows/layout-spec-maintainer.lock.yml,.github/workflows/layout-spec-maintainer.lock.yml,True,True,None,merged,error,error,1,severity
3,aidev_v5:stack-auth/stack-auth:2879676348:.git...,actionlint,A_action,introduced,"the runner of ""actions/setup-node@v3"" action i...",jobs.test.steps[name:Setup Node.js v20]#13d0b1...,16.0,"the runner of ""actions/setup-node@v3"" action i...",None,False,...,.github/workflows/restart-dev-and-test.yaml,.github/workflows/restart-dev-and-test.yaml,False,True,None,merged,error,error,2,severity
4,aidev_v5:mxcl/Path.swift:3892171796:.github/wo...,actionlint,A_action,introduced,"the runner of ""actions/checkout@v1"" action is ...",jobs.linux.steps[uses:actions/checkout]#85f9e3...,53.0,"the runner of ""actions/checkout@v1"" action is ...",None,True,...,.github/workflows/ci.yml,.github/workflows/ci.yml,True,True,None,merged,error,error,1,severity
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56905,aidev_v5:openai/codex:3623898119:.github/workf...,zizmor,Z_unsound_ternary,introduced,unsound pseudo-ternary expression,workflow,334.0,'',None,True,...,.github/workflows/rust-ci.yml,.github/workflows/rust-ci.yml,True,True,None,merged,Low,Low,2,merge_status
56906,aidev_v5:PostHog/posthog:3934931834:.github/wo...,zizmor,Z_unsound_ternary,introduced,unsound pseudo-ternary expression,workflow,1200.0,'',None,True,...,.github/workflows/ci-backend.yml,.github/workflows/ci-backend.yml,True,True,None,merged,Low,Low,1,merge_status
56907,aidev_v5:ublue-os/bluefin:3278838761:.github/w...,zizmor,Z_unsound_ternary,introduced,unsound pseudo-ternary expression,workflow,52.0,'',None,False,...,.github/workflows/notify-docs-release.yml,.github/workflows/notify-docs-release.yml,False,True,None,merged,Low,Low,3,merge_status
56908,aidev_v5:tenstorrent/tt-metal:3762393757:.gith...,zizmor,Z_unsound_ternary,introduced,unsound pseudo-ternary expression,workflow,93.0,'',None,True,...,.github/workflows/wheels.yaml,.github/workflows/wheels.yaml,True,True,None,merged,Low,Low,1,merge_status


In [57]:
# Explain each unique case once. All memberships remain in targeted_sampling_df.
explanation_cases_df = (
    targeted_sampling_df
    .drop(columns="sampling_target")
    .drop_duplicates(["pair_id", "oracle", "rule_id"])
    .reset_index(drop=True)
)

explanation_cases_df


,pair_id,oracle,rule_id,status,detail,anchor,line,evidence,before_evidence,needs_review,...,head_sha,before_path,after_path,before_exists,after_exists,metadata_error,merge_status,severity,all_severities,finding_instances
0,aidev_v5:Azure/azure-sdk-for-python:3500258934...,actionlint,A_action,introduced,"the runner of ""actions/checkout@v2"" action is ...",jobs.verify-azpysdk-checks.steps[uses:actions/...,59.0,"the runner of ""actions/checkout@v2"" action is ...",None,True,...,e8a5c2091d90247873b9a082d9621c4ccf876d81,.github/workflows/azure-sdk-tools.yml,.github/workflows/azure-sdk-tools.yml,True,True,None,open,error,error,2
1,aidev_v5:eclipse-rdf4j/rdf4j:3462373328:.githu...,actionlint,A_action,introduced,"the runner of ""actions/setup-node@v3"" action i...",jobs.copilot-setup.steps[name:Install Node.js ...,27.0,"the runner of ""actions/setup-node@v3"" action i...",None,False,...,fdaf0eb2b0f9d873ebac194b103d7aabfc58238a,.github/workflows/copilot-setup-steps.yml,.github/workflows/copilot-setup-steps.yml,False,True,None,closed_unmerged,error,error,1
2,aidev_v5:github/gh-aw:3757286500:.github/workf...,actionlint,A_action,introduced,"specifying action ""actions/setup"" in invalid f...",jobs.safe_outputs.steps[name:Setup Scripts]#35...,6212.0,"specifying action ""actions/setup"" in invalid f...",None,True,...,8a24e1eecb4f4bcd3cd7c0013c44ec30e61d92c0,.github/workflows/layout-spec-maintainer.lock.yml,.github/workflows/layout-spec-maintainer.lock.yml,True,True,None,merged,error,error,1
3,aidev_v5:stack-auth/stack-auth:2879676348:.git...,actionlint,A_action,introduced,"the runner of ""actions/setup-node@v3"" action i...",jobs.test.steps[name:Setup Node.js v20]#13d0b1...,16.0,"the runner of ""actions/setup-node@v3"" action i...",None,False,...,ec3d81f4abfa6e265a8be3e0b0e4b453794dd20a,.github/workflows/restart-dev-and-test.yaml,.github/workflows/restart-dev-and-test.yaml,False,True,None,merged,error,error,2
4,aidev_v5:mxcl/Path.swift:3892171796:.github/wo...,actionlint,A_action,introduced,"the runner of ""actions/checkout@v1"" action is ...",jobs.linux.steps[uses:actions/checkout]#85f9e3...,53.0,"the runner of ""actions/checkout@v1"" action is ...",None,True,...,e40e4ec0224a349061bd74e14226cf79dffd2a44,.github/workflows/ci.yml,.github/workflows/ci.yml,True,True,None,merged,error,error,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20303,aidev_v5:microsoft/aspire:3373438309:.github/w...,zizmor,Z_unsound_ternary,introduced,unsound pseudo-ternary expression,workflow,230.0,'',None,True,...,b94c080e7fd671923288ee749cbb935a6fdcdfb4,.github/workflows/run-tests.yml,.github/workflows/run-tests.yml,True,True,None,closed_unmerged,Low,Low,2
20304,aidev_v5:tenstorrent/tt-metal:3762393757:.gith...,zizmor,Z_unsound_ternary,introduced,unsound pseudo-ternary expression,workflow,93.0,'',None,True,...,554d441bafabab858061b2eb0c44a1774844e320,.github/workflows/wheels.yaml,.github/workflows/wheels.yaml,True,True,None,merged,Low,Low,1
20305,aidev_v5:tenstorrent/tt-metal:3934939393:.gith...,zizmor,Z_unsound_ternary,introduced,unsound pseudo-ternary expression,workflow,120.0,'',None,False,...,acfaba9e438fcb8776e60471ec88812a797064bf,.github/workflows/release-demo-tests-impl.yaml,.github/workflows/release-demo-tests-impl.yaml,False,True,None,merged,Low,Low,1
20306,aidev_v5:PostHog/posthog:4031546771:.github/wo...,zizmor,Z_unsound_ternary,introduced,unsound pseudo-ternary expression,workflow,1200.0,'',None,True,...,a15f6823a6b26e2d550efb8475e7e526ac9e197f,.github/workflows/ci-backend.yml,.github/workflows/ci-backend.yml,True,True,None,open,Low,Low,1


In [58]:
# Count memberships BEFORE deduplication: one bucket per rule x dimension.
membership_counts = targeted_sampling_df.groupby(
    ["oracle", "rule_id", "sampling_target"]
).size()

assert membership_counts.max() <= sample_size
n_rules = reduced_sampling_df[["oracle", "rule_id"]].drop_duplicates().shape[0]
assert len(membership_counts) == n_rules * len(target_categories)
assert not explanation_cases_df.duplicated(["pair_id", "oracle", "rule_id"]).any()

print("Rule x dimension combinations:", len(membership_counts))
print("Sample memberships:", len(targeted_sampling_df))
print("Unique cases to explain:", len(explanation_cases_df))
membership_counts


Rule x dimension combinations: 240
Sample memberships: 56910
Unique cases to explain: 20308


oracle      rule_id            sampling_target
actionlint  A_action           agent_name         384
                               change_type        384
                               merge_status       384
                               repo               384
                               severity           384
                                                 ... 
zizmor      Z_unsound_ternary  agent_name          16
                               change_type         16
                               merge_status        16
                               repo                16
                               severity            16
Length: 240, dtype: int64

In [62]:
explanation_cases_df.to_parquet(
    "Data/full_run/explanation_cases.parquet",
    engine="pyarrow", index=False)

### Export evidence for LLM review

This packages existing data only; it does not call an LLM or rerun scanners. Each case links to a full workflow snapshot and a rule definition. All introduced finding instances are included. Agent and sample metadata are saved separately. Read the export manifest for missing evidence before judging.


In [63]:
# The helper keeps file-packaging details out of the sampling cells.
import sys
export_module_dir = Path.cwd() if (Path.cwd() / "case_export.py").exists() else Path.cwd() / "Paper A"
sys.path.insert(0, str(export_module_dir))
from case_export import export_cases

export_dir = results_dir / "llm_review_export"
export_manifest = export_cases(
    explanation_cases_df, targeted_sampling_df, sampling_df,
    results_dir, export_dir,
)
print("Export folder:", export_dir)
export_manifest


Export folder: E:\!My Files\CS\Grammar\WorkflowResearch_Generated\paper_a_outputs\llm_review_export


{'requested_cases': 20308,
 'exported_cases': 20308,
 'exported_snapshots': 13498,
 'rules': 48,
 'errors': 0,
 'source_sha256': {'custom': '8905f72c228476b49796d2ddcc9374c6013a0a8c337e60543ec0a71a26488147',
  'adapters': '691785d830385edf4280be0c464422293ecfcbf9c04c3a851c74ecf7e8867c99',
  'matching': '116efd70768578276c11b32968a9a423abda7eac74dc9b7406b975845938959d'},
 'prompt_sha256': '271b51ccaa25336cc46a74e2b1507e821775ed49909e131d8dd9298255043bc7',
 'schema_sha256': 'fb8f60b9dffe8fa871f2d268cf40924a2051687e68fe6f5d9df9ff8f0d3f0384',
 'case_index': 'case_index.json',
 'status': 'complete',
 'notes': 'No model calls. Metadata is withheld from judging, but repository/agent identity may still be inferable from workflow text. Current checker source is archived; hashes do not prove historical source equality at scan time.'}